Abruf von externen Daten

In [20]:
import ssl
from io import StringIO
from urllib.request import urlopen

import certifi
import pandas as pd
from lxml import html as lxml_html

In [42]:
url = "https://schuldatenbank.sachsen.de/index.php?id=110&institution_key=4233037"

context = ssl.create_default_context(cafile=certifi.where())

In [36]:
with urlopen(url, timeout=10, context=context) as response:
    html = response.read().decode("utf-8")

tabellen = pd.read_html(StringIO(html))

len(tabellen)

7

Struktur der gefundenen Tabellen ansehen (Ausgabe wird von VScode abgeschnitten):

In [23]:
for i, tabelle in enumerate(tabellen):
    print(f"Tabelle {i}:")
    print(tabelle.head(3))
    print()

Tabelle 0:
  Klassenstufe  Durchschnittliche Klassenstärke*  Anzahl
0            5                              24.5      49
1            6                              27.5      55
2            7                              24.3      73

Tabelle 1:
  Klassenstufe  Durchschnittliche Klassenstärke*  Anzahl
0            5                              27.0      54
1            6                              26.0      52
2            7                              23.0      69

Tabelle 2:
  Art der Beschäftigung Beschäftigungsumfang
0        Hausmeister/in         42 h / Woche
1     Sachbearbeiter/in         35 h / Woche

Tabelle 3:
              Art der Beschäftigung  \
0  Sozialarbeiter / Sozialpädagogen   
1                          Sonstige   
2          Berufseinstiegsbegleiter   

                                           Tätigkeit  \
0                              Schulsozialarbeiterin   
1                                     Schulassistenz   
2  Berufseinstiegsbegleiterin für uns

In allen 9 Tabelle nach "Realschulabschluss" suchen:

In [30]:
for i, tabelle in enumerate(tabellen):
    if tabelle.astype(str).apply(
        lambda spalte: spalte.str.contains(
            "Realschulabschluss",
            case=False,
            na=False,
        ).any()
    ).any():
        print(f"Treffer in Tabelle {i}")

Treffer in Tabelle 6


Ergebnis: Tabelle 6 enthält damit die Schulabschlüsse.

In [29]:
tabellen[6]

,Abschluss,Gesamt,Anteil in %
0,Abgangszeugnis,8,13.6
1,Hauptschulabschluss,4,6.8
2,Qualifizierendem Hauptschulabschluss,2,3.4
3,Realschulabschluss,45,76.3


Schuljahr im geladenen HTML suchen, um dann gezielt diese Schuljahre an allen 380 Schulen abzurufen.

In [26]:
"2024/25" in html, "2023/24" in html

(True, False)

Ergebnis: Im aktuell geladenen HTML steckt nur 2024/25, nicht 2023/24.

In [27]:
position = html.find("2024/25")

print(html[position - 500 : position + 500])

ontent-1-1">
<ul class="nav nav-tabs" role="tablist">
<li role="presentation">
<a href="#tablist-1_tab-1" class="tab-head" aria-controls="tablist-1_tab-1" role="tab" data-toggle="tab">
2026/27
</a>
</li>
<li role="presentation" class="active">
<a href="#tablist-1_tab-2" class="tab-head" aria-controls="tablist-1_tab-2" role="tab" data-toggle="tab">
2025/26
</a>
</li>
<li role="presentation">
<a href="#tablist-1_tab-3" class="tab-head" aria-controls="tablist-1_tab-3" role="tab" data-toggle="tab">
2024/25
</a>
</li>
</ul>
<div class="tab-content">
<div role="tabpanel" class="tab-pane"id=tablist-1_tab-1>
<h2>Nach Klassenstufe</h2>
<p>Es liegen keine Daten vor</p>
<h2>Klassenwiederholung</h2>
Es liegen keine Daten vor
</div>
<div role="tabpanel" class="tab-pane active"id=tablist-1_tab-2>
<h2>Nach Klassenstufe</h2>
<table class='table-center'>
<thead>
<tr>
<th>Klassenstufe</th>
<th>Durchschnittliche Klassenstärke*</th>
<th>Anzahl</th>
</tr>
</thead>
<tbody>
<tr>
<td>5</td>
<td>24.5</td>
<td>

In [37]:
baum = lxml_html.fromstring(html)

for tab in baum.xpath('//div[contains(@class, "tab-pane")]'):
    text = " ".join(tab.itertext())

    if "Realschulabschluss" in text:
        print("Tab-ID:", tab.get("id"))
        print(text[:1000])

Tab-ID: tablist-2_tab-3

 Nach Abschluss 
Mit Beendigung der Vollzeitschulpflicht mit:
 
 
 
 Abschluss 
 Gesamt 
 Anteil in % 
 
 
 
 Abgangszeugnis 
 1 
 100.0 
 
 
 Hauptschulabschluss 
 0 
 0.0 
 
 
 Qualifizierendem Hauptschulabschluss 
 0 
 0.0 
 
 
 Realschulabschluss 
 0 
 0.0 
 
 
 
 Quelle: Statistisches Landesamt Amtliche Schulstatistik 2024/2025, Stand: 02.10.2024 
 Nach Prüfungsergebnis 
 
 
 
 Prüfungsnotenmittelwert 
 Schule 
 Oberschulen im Aufsichtsbezirk Leipzig 
 Oberschulen in Sachsen 
 
 
 
 
 Mathematik 
 0.0 
 4.2 
 3.9 
 
 
 Deutsch 
 0.0 
 3.1 
 3.1 
 
 
 Bestehensquote in % 
 0.0 
 95.0 
 96.0 
 
 
 
 Quelle: Meldung durch Schulaufsichtsbehörden, Stand: 27.06.2025 



Abschlusstabelle damit technisch eindeutig gefunden: Sie liegt in tablist-2_tab-3. Nächster Schritt: Schuljahre von tablist-2 auslesen: 

In [38]:
for link in baum.xpath('//a[starts-with(@href, "#tablist-2_tab-")]'):
    print(link.get("href"), "→", link.text_content().strip())

#tablist-2_tab-1 → 2026/27
#tablist-2_tab-2 → 2025/26
#tablist-2_tab-3 → 2024/25


Jetzt testen, ob der jeweilige Schuljahres-Tab Abschlussdaten enthält:

In [34]:
tab_zu_schuljahr = {
    "tablist-2_tab-1": "2026/27",
    "tablist-2_tab-2": "2025/26",
    "tablist-2_tab-3": "2024/25",
}

for tab_id, schuljahr in tab_zu_schuljahr.items():
    tab = baum.get_element_by_id(tab_id)
    text = " ".join(tab.itertext())

    hat_abschlussdaten = "Realschulabschluss" in text

    print(schuljahr, "→", hat_abschlussdaten)

2026/27 → False
2025/26 → False
2024/25 → True


Jetzt zur Sicherheit noch beispielhaft eine zweite Schule testen und dafür gezielt zwei IDs aus der eigenen schulen(1).csv verwenden. z. B. die Aktive Schule Leipzig – Freie Oberschule; sie ist in der Schuldatenbank eindeutig als freie Trägerschaft ausgewiesen. Ihr Dienststellenschlüssel lautet 4431615:

Prüfe jetzt bei dieser freien Oberschule, ob für 2024/25 tatsächlich Abschlussdaten vorhanden sind:

In [39]:
for tab_id, schuljahr in {
    "tablist-2_tab-1": "2026/27",
    "tablist-2_tab-2": "2025/26",
    "tablist-2_tab-3": "2024/25",
}.items():
    tab = baum.get_element_by_id(tab_id)
    text = " ".join(tab.itertext())

    hat_abschlussdaten = "Realschulabschluss" in text

    print(schuljahr, "→", hat_abschlussdaten)

2026/27 → False
2025/26 → False
2024/25 → True


Damit ist der technische Test an einer öffentlichen und einer freien Oberschule erfolgreich:

beide Seiten verwenden dieselbe Schuljahrstruktur,
Abschlussdaten sind jeweils für 2024/25 vorhanden,
für 2025/26 und 2026/27 noch nicht,
die Anzahl der HTML-Tabellen variiert jedoch (9 vs. 7).

## Nächster Schritt: Abschlussdaten aus 2024/25 gezielt mit Hilfe einer Funktion extrahieren

In [40]:
def extrahiere_abschluesse_2024_25(baum):
    tab = baum.get_element_by_id("tablist-2_tab-3")

    tabellen_im_tab = tab.xpath(".//table")

    if not tabellen_im_tab:
        return None

    html_tabelle = lxml_html.tostring(
        tabellen_im_tab[0],
        encoding="unicode",
    )

    return pd.read_html(StringIO(html_tabelle))[0]

Die Funktion macht drei Dinge:

Sie greift direkt auf tablist-2_tab-3 → 2024/25 zu.
Sie sucht nur innerhalb dieses Tabs nach einer Tabelle.
Wenn keine Tabelle vorhanden ist, gibt sie None zurück, statt einen Fehler zu erzeugen.

Testen der Funktion:

In [41]:
df_test_abschluesse = extrahiere_abschluesse_2024_25(baum)

df_test_abschluesse

,Abschluss,Gesamt,Anteil in %
0,Abgangszeugnis,1,100.0
1,Hauptschulabschluss,0,0.0
2,Qualifizierendem Hauptschulabschluss,0,0.0
3,Realschulabschluss,0,0.0


Jetzt Funktion an der ersten Testschule gegenprüfen (dazu wurde die url oben geändert!):

In [43]:
with urlopen(url, timeout=10, context=context) as response:
    html = response.read().decode("utf-8")

baum = lxml_html.fromstring(html)

df_test_abschluesse = extrahiere_abschluesse_2024_25(baum)

df_test_abschluesse

,Abschluss,Gesamt,Anteil in %
0,Abgangszeugnis,8,13.6
1,Hauptschulabschluss,4,6.8
2,Qualifizierendem Hauptschulabschluss,2,3.4
3,Realschulabschluss,45,76.3


Test erfolgreich --> es kann zum automatisierten Abruf übergegangen werden. Zuvor soll noch die bereits exportierte schulen (1).csv im Notebook eingelesen und ihre Struktur geprüft werden. Sie soll später unsere Liste der abzurufenden Schul-IDs und die Trägerinformationen liefern.

## Einlesen der Daten aus oberschulen_schuldatenbank.csv (ehemalig schulen (1)):

In [45]:
df_schulen = pd.read_csv(
    "../data/raw/oberschulen_schuldatenbank.csv"
)

display(df_schulen.head())
df_schulen.info()

,id,name,owner_id,owner_abbreviation,owner_name,owner_type
0,280,"1. Oberschule Großenhain ""Am Kupferberg""",402,NaN,Stadt Großenhain,15
1,2158,1. Oberschule Kamenz,833,NaN,Landkreis Bautzen,16
2,485,"101. Oberschule ""Johannes Gutenberg""",874,NaN,Stadt Dresden,14
3,3763,107. Oberschule Dresden,874,NaN,Stadt Dresden,14
4,584,116. Oberschule Dresden,874,NaN,Stadt Dresden,14


<class 'pandas.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   id                  380 non-null    int64
 1   name                380 non-null    str  
 2   owner_id            380 non-null    int64
 3   owner_abbreviation  93 non-null     str  
 4   owner_name          380 non-null    str  
 5   owner_type          380 non-null    int64
dtypes: int64(3), str(3)
memory usage: 17.9 KB


Vorabprüfung. ob Schulen doppelt enthalten sind:

In [46]:
df_schulen["id"].duplicated().sum()

np.int64(0)

Prüfung der vorhandenen Trägerarten:

In [48]:
df_schulen["owner_type"].value_counts().sort_index()

owner_type
11     1
12    42
13    73
14    72
15    87
16    12
18     1
20    20
21    16
22    36
29    20
Name: count, dtype: int64

Ergebnis: 
11 Trägerarten Auffällig: 
11-18-->288 Schulen
20-29--> 92 Schulen (vermutlich erste Gruppe öffentlich und zweite Gruppe freier Träger)

Nächster Schritt: Beispiele ansehen für "owner_type" und "owner_name"

In [49]:
df_schulen.groupby("owner_type")["owner_name"].first()

owner_type
11    Freistaat Sachsen - Staatsministerium für Wiss...
12                                  Gemeinde Kodersdorf
13                          Stadt Ebersbach-Neugersdorf
14                                        Stadt Dresden
15                                     Stadt Großenhain
16                                    Landkreis Bautzen
18                           Schulverband Treuener Land
20    Kinderbetreuungs- und Bildungsträger St. Egidi...
21                                Bistum Dresden-Meißen
22                                       epharisto e.V.
29    Gemeinnützige Gesellschaft Semper Bildungswerk...
Name: owner_name, dtype: str

Plausibilitätscheck mit deinem bestehenden Datensatz zur Zahl öffentlicher und freier Oberschulen "oberschulen_sachsen_oeffentlich_vs_frei.csv"

Öffentliche und freie Oberschulen desselben Schuljahres nebeneinander:

In [52]:
df_schulzahlen.loc[
    df_schulzahlen["schuljahr"] == "2024/2025",
    ["schuljahr", "traegerschaft", "schulen"]
]

,schuljahr,traegerschaft,schulen
32,2024/2025,öffentlich,286
66,2024/2025,frei,86


Owner_id prüfen um eine Zuordnungsmöglichkeit von öffentlicher und freier Schule zu finden:

In [53]:
df_schulen.loc[0, ["name", "owner_id", "owner_name", "owner_type"]]

name          1. Oberschule Großenhain "Am Kupferberg"
owner_id                                           402
owner_name                            Stadt Großenhain
owner_type                                          15
Name: 0, dtype: object

In [54]:
owner_id = 402

owner_url = (
    "https://schuldatenbank.sachsen.de/"
    f"index.php?id=20&k_tid={owner_id}"
)

with urlopen(owner_url, timeout=10, context=context) as response:
    owner_html = response.read().decode("utf-8")

print("öffentlicher Träger" in owner_html)
print("freier Träger" in owner_html)

True
False


Ergebnis: Die owner_id führt uns also tatsächlich zur offiziellen Trägerseite, und die Rechtsstellung lässt sich aus dem HTML auslesen.

Jetzt owner_id testen die zu einer freien Schule passt:

In [55]:
df_schulen.loc[
    df_schulen["owner_type"].isin([20, 21, 22, 29]),
    ["name", "owner_id", "owner_name", "owner_type"],
].head()

,name,owner_id,owner_name,owner_type
32,Achatschule St. Egidien-Oberschule mit Berufso...,6046,Kinderbetreuungs- und Bildungsträger St. Egidi...,20
34,AHFOberschule Markkleeberg (Schule in freier T...,6067,AHFSchulverein e. V.,20
35,Aktive Schule Leipzig - Freie Oberschule (Schu...,6111,Aktive Schule Leipzig e.V.,20
38,Aktive Schule Dresden Oberschule des epharisto...,6142,epharisto e.V.,22
47,Bischöfliches Maria-Montessori-Schulzentrum Le...,1091,Bistum Dresden-Meißen,21


Zum Beispiel hier die Aktive Schule Leipzig e.V.

In [56]:
owner_id = 6111

owner_url = (
    "https://schuldatenbank.sachsen.de/"
    f"index.php?id=20&k_tid={owner_id}"
)

with urlopen(owner_url, timeout=10, context=context) as response:
    owner_html = response.read().decode("utf-8")

print("öffentlicher Träger" in owner_html)
print("freier Träger" in owner_html)

False
True


Funktion erstellen für eine Zuordnung von öffenlich und frei:

In [57]:
def ermittle_traegerschaft(owner_id):
    owner_url = (
        "https://schuldatenbank.sachsen.de/"
        f"index.php?id=20&k_tid={owner_id}"
    )

    with urlopen(owner_url, timeout=10, context=context) as response:
        owner_html = response.read().decode("utf-8")

    if "öffentlicher Träger" in owner_html:
        return "öffentlich"

    if "freier Träger" in owner_html:
        return "frei"

    return None

Funktionstest anhand der bekannten beiden Fälle:

In [58]:
print(ermittle_traegerschaft(402))
print(ermittle_traegerschaft(6111))

öffentlich
frei


Da ein Träger (Owner) zu mehreren Oberschulen gehören kann--> Ermitteln der Anzahl unterschiedlicher owner:

In [59]:
df_schulen["owner_id"].nunique()

250

Die Frage ist, passt die Anzahl an Schulen hier zu den 288+92=380 Schulen. Unterschiede können aber existieren, da zum einen aktueller Schulbestand (2026) und zum anderen schuljahresbezogen (2024/2025) ist.